# Genesis — live wiring map

Watch her actually run: which brain part is executing, which region is
talking to which, which daemon gears are about to fire, and the
chemistry underneath — all polled straight off the daemon's IPC socket.

**Everything here is a real signal, not an animation:**

- **Module firing** — `cpu_share` is *measured* execution. The mind's
  25 Hz sampler (`brainstem/module_sampler.py`) watches
  `sys._current_frames()` and credits each running thread to the brain
  part whose code is on the leaf frame. The mind publishes those shares
  on its heartbeat; `GET_SUBSYSTEM_TELEMETRY` serves them back.
- **Wire messages** — the thalamic bus (`thalamus/relay.py`) posts
  `HEARD` / `SPOKE` / `RECALLED` / `FELT` into a module's `task_id`
  field. Edges flash only when a real message is on the wire.
- **Gears** — `GET_INTERNALS` serves the daemon's adaptive gates
  (interoception, cpufreq, associations, dreams, sync, staleness):
  bounds, current period, countdown to next fire, smoothed demand.
- **Body** — `GET_STATE` gives neurochemistry and the generative
  self-model's `inference_signals` (surprise, free energy, precision,
  allostatic load, attunement).

**Resolution honesty.** Module telemetry refreshes on the mind's
heartbeat gate — adaptive, 5–20 s (faster under load). Body internals
update at the daemon's 200 ms tick. Default poll: 0.5 s — fine for the
body, and it never misses a module state, just batches them.


In [ ]:
import sys
import threading
import time
from collections import deque
from pathlib import Path

REPO = Path("/home/mindy/Public/genesis")
for _p in (str(REPO), str(REPO / "python")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.patches import FancyArrowPatch

from genesis_client import GenesisClient
from genesis_client.protocol import CHEM_NAMES, ZONE_NAMES
from genesis_client.types import LOBE_NAMES, MODULE_NAMES, MODULE_STATUS_NAMES

# ── the wire ────────────────────────────────────────────────────────
# thalamus/relay.py packs a message kind into the low 8 bits of a
# module's task_id. These are the *actual* kinds her regions send —
# an edge lights up only when a real one is on the wire.
KIND_NAMES = {1: "HEARD", 2: "SPOKE", 3: "RECALLED", 4: "FELT"}
KIND_ID = {v: k for k, v in KIND_NAMES.items()}
KIND_COLOR = {"HEARD": "#4fc3f7", "SPOKE": "#ffb74d",
              "RECALLED": "#ce93d8", "FELT": "#f06292"}


def msg_kind(task_id) -> int:
    return int(task_id or 0) & 0xFF


# ── the map ─────────────────────────────────────────────────────────
# Nodes are manifest modules (measured execution), plus the daemon and
# retina processes and "you". Laid out by functional neighbourhood —
# senses right, executive top, body bottom — not by skull geography.
NODES = {
    "daemon":        (0.18, 0.05),
    "subcognitive":  (0.36, 0.05),
    "dreaming":      (0.11, 0.17),
    "memory":        (0.23, 0.33),
    "emotion":       (0.41, 0.14),
    "attention":     (0.45, 0.45),
    "sensory":       (0.63, 0.28),
    "metacognition": (0.14, 0.60),
    "reasoning":     (0.42, 0.67),
    "language":      (0.65, 0.62),
    "intention":     (0.40, 0.90),
    "guardrails":    (0.68, 0.88),
    "motor":         (0.86, 0.48),
    "retina":        (0.80, 0.06),
    "you":           (0.95, 0.30),
}
MODULE_NODES = [n for n in NODES if n in set(MODULE_NAMES.values())]

# Edges are real couplings. kind=None edges are structural (mind.py's
# wiring, the region projections); kind edges are the ones the
# thalamic bus can flash — only a src==poster edge carries its
# message, then the same kind continues one hop downstream.
EDGES = [
    ("you",      "sensory",   "HEARD"),
    ("retina",   "sensory",   None),
    ("sensory",  "attention", "HEARD"),
    ("sensory",  "intention", "HEARD"),
    ("sensory",  "memory",    None),
    ("attention", "sensory",  None),
    ("attention", "language",  None),
    ("attention", "reasoning", None),
    ("intention", "attention", None),
    ("intention", "reasoning", None),
    ("intention", "language",  None),
    ("intention", "motor",     None),
    ("emotion",  "attention", "FELT"),
    ("emotion",  "intention", "FELT"),
    ("emotion",  "language",  "FELT"),
    ("memory",   "language",  "RECALLED"),
    ("memory",   "reasoning", "RECALLED"),
    ("reasoning", "memory",    None),
    ("memory",   "dreaming",  None),
    ("dreaming", "memory",    None),
    ("metacognition", "intention", None),
    ("metacognition", "language",  None),
    ("language", "motor",     "SPOKE"),
    ("motor",    "you",       "SPOKE"),
    ("guardrails", "motor",    None),
    ("guardrails", "language", None),
    ("daemon",   "emotion",   None),
    ("daemon",   "attention", None),
    ("daemon",   "dreaming",  None),
    ("subcognitive", "daemon", None),
    ("you",      "daemon",    None),
]

_EDGE_SET = {(a, b) for a, b, _ in EDGES}
KIND_EDGES = {kn: [(a, b) for a, b, c in EDGES if c == kn]
              for kn in KIND_NAMES.values()}


def _rad(a, b):
    """Curve bidirectional pairs apart so both directions stay readable."""
    return 0.22 if (b, a) in _EDGE_SET else 0.10


def draw_graph(ax, act=None, status=None, edge_heat=None, title=None):
    """One frame of the wiring map.

    act:       node -> 0..1 firing level
    status:    node -> lifecycle status string
    edge_heat: (src, dst, kind) -> 0..1 pulse remaining
    """
    act = act or {}
    status = status or {}
    edge_heat = edge_heat or {}
    ax.clear()
    ax.set_xlim(-0.03, 1.06)
    ax.set_ylim(-0.08, 1.02)
    ax.axis("off")
    ax.set_facecolor("#0d1117")

    for src, dst, kind in EDGES:
        x1, y1 = NODES[src]
        x2, y2 = NODES[dst]
        heat = edge_heat.get((src, dst, kind), 0.0)
        if heat > 0.02 and kind:
            col = KIND_COLOR[kind]
            lw, al = 1.0 + 3.0 * heat, 0.40 + 0.55 * heat
        elif kind:
            col, lw, al = KIND_COLOR[kind], 1.0, 0.32  # tinted even at rest
        else:
            col, lw, al = "#8b95a7", 0.9, 0.20
        ax.add_patch(FancyArrowPatch(
            (x1, y1), (x2, y2),
            connectionstyle=f"arc3,rad={_rad(src, dst)}",
            arrowstyle="-|>" if kind else "-",
            mutation_scale=9, lw=lw, color=col, alpha=al, zorder=1,
        ))
        if kind:
            r = _rad(src, dst)
            mx = (x1 + x2) / 2 + (y2 - y1) * r * 0.7
            my = (y1 + y2) / 2 - (x2 - x1) * r * 0.7
            ax.text(mx, my, kind, fontsize=5.5, color=col, alpha=0.75,
                    ha="center", va="center", zorder=2)

    for name, (x, y) in NODES.items():
        a = min(1.0, max(0.0, act.get(name, 0.0)))
        st = status.get(name, "off")
        if st == "error":
            face = "#e53935"
        elif st in ("stopped", "stopping"):
            face = "#11151c"
        elif a > 0.001:
            face = plt.cm.inferno(0.20 + 0.80 * a)
        elif st == "running":
            face = "#2f3b52"
        else:
            face = "#232a36"
        size = 1400 + 2600 * a
        ring = "#ffffff" if st == "running" else "#5a6474"
        if a > 0.30:  # firing halo
            ax.scatter([x], [y], s=size * 2.4, facecolors="none",
                       edgecolors=[face], alpha=0.4 * a, linewidths=2.0,
                       zorder=2)
        ax.scatter([x], [y], s=size, c=[face], edgecolors=ring,
                   linewidths=1.2, zorder=3)
        pct = f" {a * 100:.0f}%" if (a > 0.01 and name in MODULE_NODES) else ""
        ax.text(x, y - 0.058, name + pct, ha="center", va="top",
                fontsize=8, color="#c9d1d9", zorder=4)

    if title:
        ax.set_title(title, color="#c9d1d9", fontsize=10, loc="left")


def show_wiring():
    """The static anatomy — works with no daemon at all."""
    fig, ax = plt.subplots(figsize=(11, 8.5), facecolor="#0d1117")
    draw_graph(ax, title="Genesis — wiring (static anatomy)")
    plt.show()


In [ ]:
# The anatomy alone — runs with no daemon. This is the "what's wired
# to what" reference; the live cells below light it up.
show_wiring()


In [ ]:
# ── connect ─────────────────────────────────────────────────────────
# MODE:
#   "auto"    — attach to the live socket if she is running, else sandbox
#   "live"    — only attach to the running instance (refuse to spawn)
#   "sandbox" — always a throwaway daemon in a temp dir, never her state
# SPAWN_MIND (sandbox only): also boot an in-process Mind on the
# sandbox socket so the module layer actually fires. Boots in the
# background; offline=True keeps the learner from touching the net.
MODE = "auto"
SOCKET = None            # explicit socket path, else the default data dir
SPAWN_MIND = True

CLIENT = None
MIND = None
_SANDBOX = None

from infrastructure.config import default_data_dir


def _live_socket():
    return str(Path(SOCKET) if SOCKET else default_data_dir() / "genesis.sock")


def connect(mode=MODE, spawn_mind=SPAWN_MIND):
    global CLIENT, MIND, _SANDBOX
    sock = _live_socket()
    if mode in ("live", "auto") and Path(sock).exists():
        c = GenesisClient(sock)
        c.connect()
        c.ping()
        CLIENT, _SANDBOX = c, None
        print(f"live: attached to {sock}")
        print("note: the module layer lights up only while her mind is "
              "running — start her with ./run.sh first")
        return CLIENT
    if mode == "live":
        print(f"live requested but nothing at {sock} — is she running?")
        return None

    from eval.harness import find_daemon_binary, isolated_daemon
    assert find_daemon_binary(), "genesis-daemon not built — cargo build --release"
    _SANDBOX = isolated_daemon()
    CLIENT = _SANDBOX.__enter__()
    print(f"sandbox: throwaway daemon on {CLIENT.socket_path}")

    if spawn_mind:
        import os
        # The steward key lives under ~/.config/genesis by default —
        # pin it inside the temp dir so the sandbox can never touch
        # the real credential or her real stamp.
        _old_key = os.environ.get("GENESIS_STEWARD_KEY")
        os.environ["GENESIS_STEWARD_KEY"] = str(
            Path(CLIENT.socket_path).parent / "steward.key")

        def _boot():
            global MIND
            try:
                from mind import Mind
                m = Mind(socket_path=CLIENT.socket_path, offline=True)
                m.start()
                MIND = m
            except Exception as e:
                print(f"  mind failed to boot — body panels still work: {e}")
            finally:
                if _old_key is None:
                    os.environ.pop("GENESIS_STEWARD_KEY", None)
                else:
                    os.environ["GENESIS_STEWARD_KEY"] = _old_key

        threading.Thread(target=_boot, daemon=True).start()
        print("sandbox: mind booting in background — "
              "module nodes appear as it registers them")
    return CLIENT


connect()


In [ ]:
# ── the monitor ─────────────────────────────────────────────────────
PROC_NODE = {0: "daemon", 2: "retina"}          # LOBE id -> map node
ROWS = [MODULE_NAMES[i] for i in sorted(MODULE_NAMES)] + [
    "proc:daemon", "proc:cognitive", "proc:retina"]
GATE_ORDER = ["intero", "cpufreq", "assoc", "dream", "sync", "staleness"]
PULSE_S = 3.0                                    # edge flash decay


def _f(x, default=0.0):
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


class LiveMap:
    """Polls the daemon feeds and renders the wiring live.

    Signals per frame:
      GET_SUBSYSTEM_TELEMETRY  module cpu_share/status/task_id + process cpu
      GET_INTERNALS            adaptive gears, lease, tick, volition
      GET_STATE                chemistry, phase, zone, inference signals
      GET_FIELD_PROBE          substrate carrier (coherence, freq)
      GET_MEMORY_STATS         stm / ltm counts
    """

    def __init__(self, client, interval=0.5, history=240):
        self.client = client
        self.interval = interval
        self.history = history
        self.stop_event = threading.Event()
        self.hist = {r: deque([0.0] * history, maxlen=history) for r in ROWS}
        self.events = deque(maxlen=9)
        self.edge_exp = {}        # (src,dst,kind) -> expiry epoch
        self.node_flash = {}      # node -> expiry epoch
        self.gates = {}           # gate name -> {period_ms,next_in_ms,...}
        self._last_task = {}      # module name -> last task_id seen
        self._gate_next = {}      # gate name -> last next_in_ms
        self._last_phase = self._last_zone = None
        self._last_hb = None
        self._surprised = False
        self.latest = {}
        self.err = None
        self._thread = None
        self.fig = None

    # ── events / ticker ────────────────────────────────────────
    def event(self, text):
        self.events.appendleft(f"{time.strftime('%H:%M:%S')}  {text}")

    # ── polling ────────────────────────────────────────────────
    def _safe(self, fn):
        try:
            return fn(timeout=5.0)
        except Exception as e:
            self.err = repr(e)
            return None

    def poll(self):
        now = time.time()
        rep = self._safe(self.client.get_subsystem_telemetry)
        internals = self._safe(self.client.get_internals) or {}
        state = self._safe(self.client.get_state)
        probe = self._safe(self.client.get_field_probe)
        mems = self._safe(self.client.get_memory_stats)
        self.latest = dict(rep=rep, internals=internals, state=state,
                           probe=probe, mems=mems)

        # module telemetry -> raster + wire messages
        if rep:
            for m in rep.modules:
                name = m.module_name
                self.hist[name].append(m.cpu_share)
                k = msg_kind(m.task_id)
                # First sighting is baseline, not a message: a dead
                # mind's task_id stays in the manifest, and without
                # this gate every attach would replay its last post.
                prev = self._last_task.get(name)
                self._last_task[name] = m.task_id
                if k and prev is not None and m.task_id != prev:
                    kn = KIND_NAMES[k]
                    self.event(f"{name} · {kn}")
                    # the posting module's own projections fire now;
                    # the same kind continues one hop downstream
                    for s, d in KIND_EDGES.get(kn, []):
                        if s == name:
                            self.edge_exp[(s, d, kn)] = now + PULSE_S
                            for s2, d2 in KIND_EDGES.get(kn, []):
                                if s2 == d:
                                    self.edge_exp[(s2, d2, kn)] = (
                                        now + PULSE_S + 0.35)
                    self.node_flash[name] = now + PULSE_S
            for s in rep.subsystems:
                self.hist[f"proc:{s.subsystem_name}"].append(s.cpu)

        # gears -> fire detection: next_in jumps up when it reschedules
        self.gates = {}
        for k, v in internals.items():
            p = k.split(".")
            if len(p) == 3 and p[0] == "gate":
                self.gates.setdefault(p[1], {})[p[2]] = v
            elif len(p) == 4 and p[0] == "auto" and p[1] == "gate":
                self.gates.setdefault("auto·" + p[2], {})[p[3]] = v
        for g, d in self.gates.items():
            nxt = _f(d.get("next_in_ms"))
            prev = self._gate_next.get(g)
            if prev is not None and nxt > prev + 100:
                self.event(f"gear {g} fired")
                self.node_flash["daemon"] = now + 1.0
            self._gate_next[g] = nxt

        # phase / zone / heartbeat transitions
        if state:
            ph = state.phase_name
            zn = ZONE_NAMES.get(state.cognitive_zone,
                                str(state.cognitive_zone))
            if self._last_phase is not None and ph != self._last_phase:
                self.event(f"phase → {ph}")
            if self._last_zone is not None and zn != self._last_zone:
                self.event(f"zone → {zn}")
            self._last_phase, self._last_zone = ph, zn
            if self._last_hb is not None and state.heartbeat == self._last_hb:
                self.err = "daemon heartbeat stalled"
            self._last_hb = state.heartbeat
            inf = state.inference
            if inf is not None and inf.is_surprised:
                if not self._surprised:
                    self.event(f"surprised (ema {inf.surprise_ema:.2f})")
                self._surprised = True
            else:
                self._surprised = False

    # ── per-frame activations ──────────────────────────────────
    def _activations(self):
        rep = self.latest.get("rep")
        internals = self.latest.get("internals") or {}
        state = self.latest.get("state")
        act, status = {}, {}
        if rep:
            mmax = max((m.cpu_share for m in rep.modules), default=0.0)
            norm = max(0.05, mmax)
            for m in rep.modules:
                act[m.module_name] = min(1.0, m.cpu_share / norm)
                status[m.module_name] = (
                    "error" if m.is_failed else m.status_name)
            for s in rep.subsystems:
                node = PROC_NODE.get(s.subsystem)
                if node:
                    act[node] = min(1.0, s.cpu)
                    status[node] = "running"
        act["daemon"] = max(
            act.get("daemon", 0.0),
            _f(internals.get("subcognitive_activity")))
        status.setdefault("daemon", "running" if internals else "off")
        inf = getattr(state, "inference", None) if state else None
        if inf is not None:
            act["you"] = min(1.0, max(0.0, inf.user_engagement))
            status["you"] = "running" if inf.user_engagement > 0.02 else "idle"
        for node, exp in self.node_flash.items():
            if exp > time.time():
                act[node] = max(act.get(node, 0.0), 0.45)
        return act, status

    # ── figure ─────────────────────────────────────────────────
    def _build(self):
        plt.ioff()
        self.fig = plt.figure(figsize=(16.5, 10.5), facecolor="#0d1117")
        gs = self.fig.add_gridspec(
            4, 2, width_ratios=[1.5, 1.0],
            height_ratios=[2.6, 1.2, 1.2, 1.1],
            hspace=0.38, wspace=0.16,
            left=0.03, right=0.985, top=0.93, bottom=0.05)
        self.ax_map = self.fig.add_subplot(gs[:3, 0])
        self.ax_tick = self.fig.add_subplot(gs[3, 0])
        self.ax_raster = self.fig.add_subplot(gs[0, 1])
        self.ax_gears = self.fig.add_subplot(gs[1, 1])
        self.ax_chem = self.fig.add_subplot(gs[2, 1])
        self.ax_sig = self.fig.add_subplot(gs[3, 1])
        for a in (self.ax_raster, self.ax_gears, self.ax_chem, self.ax_sig):
            a.set_facecolor("#0d1117")
            a.tick_params(colors="#8b95a7", labelsize=7)
            for sp in a.spines.values():
                sp.set_color("#30363d")
        self.ax_tick.axis("off")
        self.ax_tick.set_facecolor("#0d1117")
        self.im = self.ax_raster.imshow(
            np.zeros((len(ROWS), 2)), aspect="auto",
            cmap="inferno", vmin=0, vmax=1)
        self.ax_raster.set_yticks(range(len(ROWS)), ROWS, fontsize=6.5)
        self.ax_raster.set_xlabel("samples ago", color="#8b95a7",
                                  fontsize=7)
        self.ax_raster.set_title("firing — module cpu_share over time",
                                 color="#c9d1d9", fontsize=9)

    def _frame(self):
        now = time.time()
        state = self.latest.get("state")
        internals = self.latest.get("internals") or {}
        mems = self.latest.get("mems")
        probe = self.latest.get("probe")

        act, status = self._activations()
        heat = {k: max(0.0, min(1.0, (exp - now) / PULSE_S))
                for k, exp in self.edge_exp.items() if exp > now}

        bits = []
        if state:
            bits.append(
                f"phase {state.phase_name} · zone "
                f"{ZONE_NAMES.get(state.cognitive_zone, '?')} · "
                f"sleep {state.sleep_pressure:.2f} · "
                f"hb {state.heartbeat:,}")
        if internals:
            lease = _f(internals.get("mind_lease_idle_ms"), -1)
            bits.append(
                f"tick {internals.get('tick_count', '?')} · "
                f"lease {'none' if lease < 0 else f'{lease:.0f}ms'} · "
                f"volition {_f(internals.get('volition_vlpo')):.2f}/"
                f"{_f(internals.get('volition_lc')):.2f}")
        if self.err:
            bits.append(f"! {self.err}")
        draw_graph(self.ax_map, act=act, status=status, edge_heat=heat,
                   title="   |   ".join(bits))

        # raster
        data = np.array([list(self.hist[r]) for r in ROWS])
        self.im.set_data(data)
        self.im.set_clim(0, max(0.05, float(np.max(data))))
        n = data.shape[1]
        self.ax_raster.set_xlim(0, n)
        ticks = np.arange(0, n, max(1, n // 5))
        self.ax_raster.set_xticks(
            ticks, [f"{(n - t) * self.interval:.0f}s" for t in ticks])

        # gears — elapsed fraction + countdown + smoothed demand
        ax = self.ax_gears
        ax.clear()
        ax.set_facecolor("#0d1117")
        ordered = [g for g in GATE_ORDER if g in self.gates] + sorted(
            g for g in self.gates if g not in GATE_ORDER)
        if ordered:
            ax.barh(range(len(ordered)), [1.0] * len(ordered),
                    color="#30363d", alpha=0.5, height=0.62, zorder=0)
            for i, g in enumerate(ordered):
                d = self.gates[g]
                period = _f(d.get("period_ms"), 1.0)
                nxt = _f(d.get("next_in_ms"))
                sm = _f(d.get("smoothed"))
                frac = max(0.0, min(1.0, 1.0 - nxt / max(period, 1.0)))
                ax.barh(i, frac, color="#ffb74d", alpha=0.85,
                        height=0.62, zorder=1)
                ax.text(0.015, i,
                        f"next {nxt / 1000:.1f}s / "
                        f"{period / 1000:.1f}s · demand {sm:.2f}",
                        va="center", fontsize=6.5, color="#e8edf2",
                        zorder=2)
            ax.set_yticks(range(len(ordered)), ordered, fontsize=7)
            ax.set_ylim(-0.6, len(ordered) - 0.4)
            ax.invert_yaxis()
        ax.set_xlim(0, 1)
        ax.set_xticks([0, 1], ["wait", "fires"], fontsize=6)
        ax.set_title("daemon gears — adaptive gates",
                     color="#c9d1d9", fontsize=9)

        # neurochemistry — id order, dopamine on top
        ax = self.ax_chem
        ax.clear()
        ax.set_facecolor("#0d1117")
        if state and state.chemicals:
            names = list(state.chemicals)
            vals = [_f(state.chemicals[n]) for n in names]
            ax.barh(range(len(vals)), vals, color="#4fc3f7", alpha=0.85,
                    height=0.7)
            ax.set_yticks(range(len(names)), names, fontsize=6)
            ax.set_ylim(-0.7, len(names) - 0.3)
            ax.invert_yaxis()
        ax.set_xlim(0, 1)
        ax.tick_params(axis="x", labelsize=6)
        ax.set_title(
            f"neurochemistry   arousal {_f(getattr(state, 'arousal', 0)):.2f}"
            f"  valence {_f(getattr(state, 'valence', 0)):.2f}"
            f"  tone {_f(getattr(state, 'global_tone', 0)):.2f}",
            color="#c9d1d9", fontsize=9)

        # self-model signals
        ax = self.ax_sig
        ax.clear()
        ax.set_facecolor("#0d1117")
        inf = getattr(state, "inference", None) if state else None
        if inf is not None:
            rows = [
                ("surprise", inf.surprise_ema),
                ("free E", inf.free_energy),
                ("exp free E", inf.expected_free_energy),
                ("allostasis", inf.allostasis_load),
                ("precision", inf.precision),
                ("attunement", inf.attunement),
                ("synchrony", inf.dyadic_synchrony),
                ("maturity", inf.model_maturity),
                ("policy auth", inf.policy_authority),
            ]
            names = [r[0] for r in rows]
            vals = [max(0.0, min(1.2, _f(r[1]))) for r in rows]
            cols = ["#f06292" if v > 0.3 and n == "surprise" else "#81c784"
                    for n, v in rows]
            ax.barh(range(len(vals)), vals, color=cols, alpha=0.85,
                    height=0.7)
            ax.set_yticks(range(len(names)), names, fontsize=6)
            ax.set_ylim(-0.7, len(names) - 0.3)
        extra = []
        if mems:
            extra.append(f"stm {mems.stm_count} · ltm {mems.ltm_count}")
        if probe and getattr(probe, "samples_ok", False):
            extra.append(
                f"carrier {probe.carrier_freq:.1f}Hz · "
                f"coh {probe.carrier_coherence:.2f}")
        ax.set_xlim(0, 1.25)
        ax.set_xticks([])
        ax.set_title("self-model (inference signals)"
                     + ("   |   " + "  ·  ".join(extra) if extra else ""),
                     color="#c9d1d9", fontsize=9)

        # ticker
        self.ax_tick.clear()
        self.ax_tick.axis("off")
        self.ax_tick.set_facecolor("#0d1117")
        self.ax_tick.text(
            0.0, 0.95, "\n".join(self.events) or "listening…",
            va="top", ha="left", fontsize=7.5, color="#8b95a7",
            family="monospace", transform=self.ax_tick.transAxes)

    # ── drivers ────────────────────────────────────────────────
    def _loop(self, seconds=None):
        self._build()
        dh = display(self.fig, display_id=True)
        t0 = time.time()
        try:
            while not self.stop_event.is_set() and (
                    seconds is None or time.time() - t0 < seconds):
                t = time.time()
                try:
                    self.poll()
                    self._frame()
                    dh.update(self.fig)
                except Exception as e:
                    self.event(f"viz: {e!r}")
                dt = self.interval - (time.time() - t)
                if dt > 0:
                    self.stop_event.wait(dt)
        finally:
            self.event("monitor stopped")

    def start(self):
        """Non-blocking: watch in a thread, keep cells free for stimuli."""
        if self._thread and self._thread.is_alive():
            return self
        self.stop_event.clear()
        self.event("monitor started")
        self._thread = threading.Thread(
            target=self._loop, daemon=True, name="livemap")
        self._thread.start()
        return self

    def run(self, seconds=None):
        """Blocking variant — run(seconds=60) or interrupt to stop."""
        self.stop_event.clear()
        self.event("monitor started")
        self._loop(seconds=seconds)

    def stop(self):
        self.stop_event.set()
        t = self._thread
        if t and t.is_alive():
            t.join(timeout=3.0)


In [ ]:
# ── watch ───────────────────────────────────────────────────────────
# Runs in a background thread so the cells below stay free for
# stimuli. Blocking alternative: MAP.run(seconds=60)
MAP = LiveMap(CLIENT, interval=0.5)
MAP.start()

try:
    import ipywidgets as widgets
    _stop_btn = widgets.ToggleButton(
        value=False, description="■ stop monitor",
        button_style="danger", icon="hand-paper")
    _stop_btn.observe(
        lambda ch: MAP.stop() if ch["new"] else None, names="value")
    display(_stop_btn)
except Exception:
    print("ipywidgets unavailable — use MAP.stop() or MAP.run(seconds=N)")


In [ ]:
# ── poke her — while the map is live ────────────────────────────────
# Each call below makes something real happen. Run any line, watch the
# map answer. In live mode, talking to her in her own console works too.

CHEM_ID = {v: k for k, v in CHEM_NAMES.items()}


def say(text: str):
    """Send a conversation turn to the sandbox mind.

    Watch: sensory posts HEARD (blue edge), language+reasoning+memory
    light while she composes, then language posts SPOKE (orange edge,
    chaining out through motor to "you").
    """
    if MIND is None:
        print("no in-process mind — live mode: just talk to her, "
              "the map shows the same signals")
        return
    def _turn():
        try:
            reply = MIND.respond(text)
            print(f"genesis: {reply}")
        except Exception as e:
            print(f"respond failed: {e}")
    threading.Thread(target=_turn, daemon=True).start()


def impulse(chem="dopamine", magnitude=0.4):
    """Push one chemical — watch the chemistry bars and her phase."""
    ok = CLIENT.neuro_impulse(CHEM_ID[chem], magnitude)
    print(f"{chem} {'+' if magnitude >= 0 else ''}{magnitude}: {ok}")


def push_sleep(scale=1.0):
    """Sleep cascade — the same chemistry her /sleep control sends:
    melatonin and GABA rise while orexin, histamine, and NE withdraw.
    Watch the dream gear and the NREM/REM phase transitions."""
    for chem, mag in (("melatonin", 0.10), ("gaba", 0.10),
                      ("orexin", -0.10), ("histamine", -0.10),
                      ("norepinephrine", -0.05)):
        impulse(chem, mag * scale)


def push_wake(scale=1.0):
    """Wake cascade — the /wake arousal chemistry: orexin and
    histamine drive the LC while melatonin withdraws."""
    for chem, mag in (("orexin", 0.10), ("histamine", 0.10),
                      ("norepinephrine", 0.05), ("dopamine", 0.05),
                      ("melatonin", -0.10)):
        impulse(chem, mag * scale)


# things to try:
#   say("hello genesis")                    # a conversation turn
#   say("what are you thinking about?")     # metacognition + memory
#   impulse("dopamine", 0.5)                # reward — watch valence
#   impulse("cortisol", 0.5)                # stress — watch arousal
#   impulse("adenosine", 0.4)               # sleep pressure building
#   push_sleep(2.0)                         # drift toward sleep/dreaming
#   CLIENT.reward(0.5)                      # convenience: dopamine
#   CLIENT.dream()                          # force a dream cycle
#   CLIENT.associate()                      # force association pass


## Reading the map

**The wiring (left).** A node is bright when its code is actually
executing — the label shows its share of measured mind-work. A node in
**red** has posted an error to the manifest; dark-navy is alive-but-idle;
near-black is stopped. Edges tinted blue/orange/purple/pink are wire
paths (`HEARD`/`SPOKE`/`RECALLED`/`FELT`); they flash when a real message
crosses. The `you` node lights with her measured engagement estimate.

**Firing (top right).** A raster: every module's `cpu_share` history —
the EEG. Watch `language`/`reasoning`/`memory` sweep during a reply,
`dreaming` light in sleep, `metacognition` during introspection.

**Gears (mid right).** The daemon's adaptive gates. The amber bar fills
toward "fires"; text shows countdown / period / smoothed demand. When a
gate fires the ticker logs it — `dream` firing while she sleeps is a
dream cycle actually running.

**Chemistry (right).** All 18 effective levels, live. Arousal / valence
/ global tone in the header.

**Self-model (bottom right).** Her generative model of herself:
surprise and free energy rise when she fails to predict her own state;
precision is model confidence; attunement/synchrony measure the dyad;
policy authority is how much she's delegated to the engine.

**Ticker (bottom left).** The event log: wire messages, gear fires,
phase/zone transitions, surprises.

Known limit: module `cpu_share` and `task_id` refresh on the mind's
heartbeat gate (5–20 s adaptive), so the module layer updates in
breaths, not milliseconds — the body and gears panels are much faster.


In [ ]:
# ── teardown ────────────────────────────────────────────────────────
# Stops the monitor, stops the sandbox mind, shuts the throwaway daemon
# down and removes its temp dir. In live mode it only detaches — she
# keeps running.
MAP.stop()

if MIND is not None:
    try:
        MIND.stop()
        print("mind stopped")
    except Exception as e:
        print(f"mind stop: {e}")
    MIND = None

if _SANDBOX is not None:
    _SANDBOX.__exit__(None, None, None)
    _SANDBOX = None
    print("sandbox daemon removed")
elif CLIENT is not None:
    try:
        CLIENT.disconnect()
        print("detached — she is still running")
    except Exception:
        pass
